# morphological_quantification_2026-01-02 — 05b_day5_feature_handoff

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 05b. Day-5 Feature Handoff Table

This notebook compiles a rough draft day-5 phenotype handoff table with one row per trunk morph ID.


## Cell Guide

- `Setup`: resolve the project root, import dependencies, and define the input/output paths.
- `Settings Notes`: explain what this handoff table is for and how the draft descriptor columns are generated.
- `Load Inputs`: read the included analysis manifest plus the current organoid-level summary tables from `05c`, `05d`, `05e`, and `05f`.
- `Review Whole-Morph Shape Summary`: inspect the current whole-morph geometry summary coming from `05f`.
- `Merge Marker Features Into One Row Per Organoid`: combine FOXF1, MESP2, and PAX8 features with the shape summary.
- `Add Draft Descriptor Columns`: add rough cohort-relative size/position notes and simple bilaterality notes for handoff convenience.
- `Write And Review Handoff Table`: save the table to disk and display it inline.
- `Next Step`: later, pair this table with representative day-5 images for visual verification if needed.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().resolve().parent
elif (Path.cwd() / "notebooks").exists():
    ROOT = Path.cwd().resolve()
else:
    raise RuntimeError("Run this notebook from the project root or the notebooks/ directory.")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)


## Settings Notes

- This is a handoff notebook, not a new analysis stage. It reuses the current outputs from `04b`, `05c`, `05d`, `05e`, and `05f`.
- The core quantitative columns are the values to preserve and hand off.
- The descriptor columns are only draft convenience labels for sorting and discussion. They are not final biology calls.
- Unless otherwise noted, the handoff table now uses **mean-across-z** summaries rather than median-across-z summaries.
- Shape, LPM size, LPM posterior distance, MESP2 size, and PAX8 size are binned by cohort-relative tertiles within the currently included day-5 trunk morph cohort.
- Bilaterality draft labels use simple fixed thresholds on the current side-balance index:
  - `< 0.5`: asymmetric
  - `0.5 to < 0.8`: partially bilateral
  - `>= 0.8`: bilateral-like
- File `31` remains excluded because it was previously set aside as a background-distribution outlier.


In [ ]:
ANALYSIS_MANIFEST_PATH = ROOT / "results" / "manifests" / "analysis_manifest.tsv"
LPM_SUMMARY_TABLE_PATH = ROOT / "results" / "tables" / "05c_lpm_size_summary_by_file.tsv"
MESP2_SUMMARY_TABLE_PATH = ROOT / "results" / "tables" / "05d_mesp2_expression_summary_by_file.tsv"
PAX8_SUMMARY_TABLE_PATH = ROOT / "results" / "tables" / "05e_pax8_expression_summary_by_file.tsv"
WHOLE_MORPH_SUMMARY_TABLE_PATH = ROOT / "results" / "tables" / "05f_whole_morph_geometry_summary_by_file.tsv"

HANDOFF_TABLE_PATH = ROOT / "results" / "tables" / "05b_day5_feature_handoff_table.tsv"
HANDOFF_XLSX_PATH = ROOT / "results" / "tables" / "05b_day5_feature_handoff_table.xlsx"

FILES_PER_MONTAGE_PAGE = 2


## Load Inputs


In [ ]:
manifest_df = pd.read_csv(ANALYSIS_MANIFEST_PATH, sep="\t")
manifest_df = manifest_df.loc[
    manifest_df["include_in_analysis"].fillna(True).astype(bool)
].copy()
manifest_df = manifest_df.sort_values("file_id").reset_index(drop=True)

lpm_summary_df = pd.read_csv(LPM_SUMMARY_TABLE_PATH, sep="\t")
lpm_summary_df = lpm_summary_df.loc[
    lpm_summary_df["file_id"].isin(manifest_df["file_id"])
].copy()

mesp2_summary_df = pd.read_csv(MESP2_SUMMARY_TABLE_PATH, sep="\t")
mesp2_summary_df = mesp2_summary_df.loc[
    mesp2_summary_df["file_id"].isin(manifest_df["file_id"])
].copy()

pax8_summary_df = pd.read_csv(PAX8_SUMMARY_TABLE_PATH, sep="\t")
pax8_summary_df = pax8_summary_df.loc[
    pax8_summary_df["file_id"].isin(manifest_df["file_id"])
].copy()

shape_summary_df = pd.read_csv(WHOLE_MORPH_SUMMARY_TABLE_PATH, sep="\t")
shape_summary_df = shape_summary_df.loc[
    shape_summary_df["file_id"].isin(manifest_df["file_id"])
].copy()

print(f"Included trunk morphs: {manifest_df['file_id'].nunique()}")
print(f"LPM summary rows: {len(lpm_summary_df)}")
print(f"MESP2 summary rows: {len(mesp2_summary_df)}")
print(f"PAX8 summary rows: {len(pax8_summary_df)}")
print(f"Whole-morph summary rows: {len(shape_summary_df)}")
display(
    manifest_df[
        [
            "file_id",
            "file_name",
            "canonical_position",
            "acquisition_date",
            "size_z",
        ]
    ].head(10).style.hide(axis="index")
)


## Review Whole-Morph Shape Summary

For the handoff table, whole-morph shape now comes directly from `05f`, including both the older ellipse-based geometry summaries and the newer consensus-axis geometry summaries.


In [ ]:
display(
    shape_summary_df[
        [
            "trunk_morph_id",
            "file_id",
            "n_z_planes",
            "z_indices",
            "length_width_ratio_mean_z",
            "length_um_mean_z",
            "width_um_mean_z",
            "consensus_axis_length_um",
            "consensus_axis_perp_width_um_mean_z",
            "consensus_axis_length_width_ratio_mean_z",
        ]
    ].head(8).style.hide(axis="index").format(
        {
            "length_width_ratio_mean_z": "{:.2f}",
            "length_um_mean_z": "{:.1f}",
            "width_um_mean_z": "{:.1f}",
            "consensus_axis_length_um": "{:.1f}",
            "consensus_axis_perp_width_um_mean_z": "{:.1f}",
            "consensus_axis_length_width_ratio_mean_z": "{:.2f}",
        }
    )
)


## Merge Marker Features Into One Row Per Organoid

This step combines:
- the organoid identifiers from the analysis manifest
- the representative montage page from `04b`
- the current mean-across-z `LPM`, `MESP2`, and `PAX8` summaries from `05c`, `05d`, and `05e`
- the current whole-morph geometry summaries from `05f`

The result is one row per trunk morph ID.


In [ ]:
base_df = manifest_df[
    [
        "image_id",
        "cohort_id",
        "canonical_position",
        "file_id",
        "file_name",
        "file_path",
        "acquisition_date",
        "acquisition_batch_label",
        "size_z",
    ]
].copy()

shape_subset_df = shape_summary_df[
    [
        "file_id",
        "n_z_planes",
        "z_indices",
        "axis_display_z_index",
        "length_width_ratio_mean_z",
        "length_um_mean_z",
        "width_um_mean_z",
        "consensus_axis_length_um",
        "consensus_axis_perp_width_um_median_z",
        "consensus_axis_perp_width_um_mean_z",
        "consensus_axis_perp_width_um_max_z",
        "consensus_axis_perp_width_um_min_z",
        "consensus_axis_perp_width_um_range_z",
        "consensus_axis_length_width_ratio_median_z",
        "consensus_axis_length_width_ratio_mean_z",
        "consensus_axis_length_width_ratio_max_z",
        "consensus_axis_length_width_ratio_min_z",
        "consensus_axis_length_width_ratio_range_z",
    ]
].rename(
    columns={
        "length_width_ratio_mean_z": "length_width_ratio_mean",
        "length_um_mean_z": "major_axis_length_um_mean",
        "width_um_mean_z": "minor_axis_length_um_mean",
        "consensus_axis_perp_width_um_mean_z": "consensus_axis_perp_width_um_mean",
        "consensus_axis_length_width_ratio_mean_z": "consensus_axis_length_width_ratio_mean",
    }
)

foxf1_df = lpm_summary_df[
    [
        "file_id",
        "lpm_area_um2_mean_z",
        "lpm_fraction_mean_z",
        "lpm_posterior_extent_um_mean_z",
        "lpm_axis_p10_um_mean_z",
        "lpm_centroid_um_mean_z",
        "lpm_bilaterality_mean_z",
        "lpm_weighted_bilaterality_mean_z",
    ]
].rename(
    columns={
        "lpm_area_um2_mean_z": "foxf1_area_um2_mean",
        "lpm_fraction_mean_z": "foxf1_fraction_mean",
        "lpm_posterior_extent_um_mean_z": "foxf1_posterior_edge_distance_um_mean",
        "lpm_axis_p10_um_mean_z": "foxf1_posterior_p10_distance_um_mean",
        "lpm_centroid_um_mean_z": "foxf1_posterior_centroid_distance_um_mean",
        "lpm_bilaterality_mean_z": "foxf1_bilaterality_index_mean",
        "lpm_weighted_bilaterality_mean_z": "foxf1_weighted_bilaterality_index_mean",
    }
)
mesp2_df = mesp2_summary_df[
    [
        "file_id",
        "mesp2_area_um2_mean_z",
        "mesp2_fraction_mean_z",
        "mesp2_bilaterality_mean_z",
        "mesp2_weighted_bilaterality_mean_z",
    ]
].rename(
    columns={
        "mesp2_area_um2_mean_z": "mesp2_area_um2_mean",
        "mesp2_fraction_mean_z": "mesp2_fraction_mean",
        "mesp2_bilaterality_mean_z": "mesp2_bilaterality_index_mean",
        "mesp2_weighted_bilaterality_mean_z": "mesp2_weighted_bilaterality_index_mean",
    }
)
pax8_df = pax8_summary_df[
    [
        "file_id",
        "pax8_area_um2_mean_z",
        "pax8_fraction_mean_z",
        "pax8_bilaterality_mean_z",
        "pax8_weighted_bilaterality_mean_z",
    ]
].rename(
    columns={
        "pax8_area_um2_mean_z": "pax8_area_um2_mean",
        "pax8_fraction_mean_z": "pax8_fraction_mean",
        "pax8_bilaterality_mean_z": "pax8_bilaterality_index_mean",
        "pax8_weighted_bilaterality_mean_z": "pax8_weighted_bilaterality_index_mean",
    }
)

handoff_df = (
    base_df.merge(shape_subset_df, on="file_id", how="left")
    .merge(foxf1_df, on="file_id", how="left")
    .merge(mesp2_df, on="file_id", how="left")
    .merge(pax8_df, on="file_id", how="left")
)

handoff_df["trunk_morph_id"] = handoff_df["file_id"].astype(int).map(lambda v: f"{v:02d}")

display(
    handoff_df[
        [
            "trunk_morph_id",
            "image_id",
            "file_name",
            "length_width_ratio_mean",
            "foxf1_area_um2_mean",
            "foxf1_posterior_edge_distance_um_mean",
            "foxf1_posterior_centroid_distance_um_mean",
            "foxf1_bilaterality_index_mean",
            "foxf1_weighted_bilaterality_index_mean",
            "mesp2_area_um2_mean",
            "mesp2_bilaterality_index_mean",
            "mesp2_weighted_bilaterality_index_mean",
            "pax8_area_um2_mean",
            "pax8_bilaterality_index_mean",
            "pax8_weighted_bilaterality_index_mean",
        ]
    ].head(8).style.hide(axis="index").format(
        {
            "length_width_ratio_mean": "{:.2f}",
            "foxf1_area_um2_mean": "{:.1f}",
            "foxf1_posterior_edge_distance_um_mean": "{:.1f}",
            "foxf1_posterior_centroid_distance_um_mean": "{:.1f}",
            "foxf1_bilaterality_index_mean": "{:.2f}",
            "foxf1_weighted_bilaterality_index_mean": "{:.2f}",
            "mesp2_area_um2_mean": "{:.1f}",
            "mesp2_bilaterality_index_mean": "{:.2f}",
            "mesp2_weighted_bilaterality_index_mean": "{:.2f}",
            "pax8_area_um2_mean": "{:.1f}",
            "pax8_bilaterality_index_mean": "{:.2f}",
            "pax8_weighted_bilaterality_index_mean": "{:.2f}",
        }
    )
)


## Add Draft Descriptor Columns

These descriptor columns are only for handoff convenience.

- Shape, LPM size, LPM posterior distance, MESP2 size, and PAX8 size are binned by cohort-relative tertiles.
- Both unweighted and weighted bilaterality labels use the same side-balance thresholds described above.
- The combined text description is intentionally rough and should be treated as a discussion aid, not a final phenotype call.


In [ ]:
def tertile_note(series: pd.Series, labels):
    valid = pd.to_numeric(series, errors="coerce").dropna()
    out = pd.Series(["undetermined"] * len(series), index=series.index, dtype=object)
    if valid.empty:
        return out, np.nan, np.nan
    q1 = float(valid.quantile(1.0 / 3.0))
    q2 = float(valid.quantile(2.0 / 3.0))
    low_label, mid_label, high_label = labels
    numeric = pd.to_numeric(series, errors="coerce")
    out.loc[numeric <= q1] = low_label
    out.loc[(numeric > q1) & (numeric <= q2)] = mid_label
    out.loc[numeric > q2] = high_label
    out.loc[numeric.isna()] = "undetermined"
    return out, q1, q2


def bilaterality_note(value):
    if not np.isfinite(value):
        return "undetermined"
    if value < 0.5:
        return "asymmetric"
    if value < 0.8:
        return "partially bilateral"
    return "bilateral-like"


cutoff_rows = []

handoff_df["overall_shape_note"], q1, q2 = tertile_note(
    handoff_df["length_width_ratio_mean"],
    ("compact", "intermediate", "elongated"),
)
cutoff_rows.append(
    {
        "feature": "length/width ratio",
        "lower_cutoff": q1,
        "upper_cutoff": q2,
        "draft_labels": "compact | intermediate | elongated",
    }
)

handoff_df["foxf1_size_note"], q1, q2 = tertile_note(
    handoff_df["foxf1_area_um2_mean"],
    ("small LPM", "intermediate LPM", "large LPM"),
)
cutoff_rows.append(
    {
        "feature": "FOXF1/LPM area",
        "lower_cutoff": q1,
        "upper_cutoff": q2,
        "draft_labels": "small LPM | intermediate LPM | large LPM",
    }
)

handoff_df["foxf1_posterior_position_note"], q1, q2 = tertile_note(
    handoff_df["foxf1_posterior_edge_distance_um_mean"],
    ("posterior LPM", "mid-axis LPM", "anterior-shifted LPM"),
)
cutoff_rows.append(
    {
        "feature": "FOXF1/LPM posterior distance (um)",
        "lower_cutoff": q1,
        "upper_cutoff": q2,
        "draft_labels": "posterior LPM | mid-axis LPM | anterior-shifted LPM",
    }
)

handoff_df["mesp2_size_note"], q1, q2 = tertile_note(
    handoff_df["mesp2_area_um2_mean"],
    ("small MESP2", "intermediate MESP2", "large MESP2"),
)
cutoff_rows.append(
    {
        "feature": "MESP2 area",
        "lower_cutoff": q1,
        "upper_cutoff": q2,
        "draft_labels": "small MESP2 | intermediate MESP2 | large MESP2",
    }
)

handoff_df["pax8_size_note"], q1, q2 = tertile_note(
    handoff_df["pax8_area_um2_mean"],
    ("small IM", "intermediate IM", "large IM"),
)
cutoff_rows.append(
    {
        "feature": "PAX8/IM area",
        "lower_cutoff": q1,
        "upper_cutoff": q2,
        "draft_labels": "small IM | intermediate IM | large IM",
    }
)

handoff_df["foxf1_bilaterality_note"] = handoff_df["foxf1_bilaterality_index_mean"].map(bilaterality_note)
handoff_df["foxf1_weighted_bilaterality_note"] = handoff_df["foxf1_weighted_bilaterality_index_mean"].map(bilaterality_note)
handoff_df["mesp2_bilaterality_note"] = handoff_df["mesp2_bilaterality_index_mean"].map(bilaterality_note)
handoff_df["mesp2_weighted_bilaterality_note"] = handoff_df["mesp2_weighted_bilaterality_index_mean"].map(bilaterality_note)
handoff_df["pax8_bilaterality_note"] = handoff_df["pax8_bilaterality_index_mean"].map(bilaterality_note)
handoff_df["pax8_weighted_bilaterality_note"] = handoff_df["pax8_weighted_bilaterality_index_mean"].map(bilaterality_note)

handoff_df["draft_day5_description"] = handoff_df.apply(
    lambda row: (
        f"{row['overall_shape_note']}; "
        f"LPM {row['foxf1_size_note']}, {row['foxf1_posterior_position_note']}, {row['foxf1_bilaterality_note']}; "
        f"MESP2 {row['mesp2_size_note']}, {row['mesp2_bilaterality_note']}; "
        f"IM {row['pax8_size_note']}, {row['pax8_bilaterality_note']}"
    ),
    axis=1,
)

handoff_df["bilateral somites (y/n)"] = ""
handoff_df["neural tube with lumen (y/n)"] = ""

cutoff_df = pd.DataFrame(cutoff_rows)
display(
    cutoff_df.style.hide(axis="index").format(
        {"lower_cutoff": "{:.2f}", "upper_cutoff": "{:.2f}"}
    )
)


## Write And Review Handoff Table


In [ ]:
handoff_df = handoff_df[
    [
        "trunk_morph_id",
        "image_id",
        "file_name",
        "length_width_ratio_mean",
        "major_axis_length_um_mean",
        "minor_axis_length_um_mean",
        "consensus_axis_length_um",
        "consensus_axis_perp_width_um_mean",
        "consensus_axis_length_width_ratio_mean",
        "overall_shape_note",
        "foxf1_area_um2_mean",
        "foxf1_fraction_mean",
        "foxf1_posterior_edge_distance_um_mean",
        "foxf1_posterior_p10_distance_um_mean",
        "foxf1_posterior_centroid_distance_um_mean",
        "foxf1_bilaterality_index_mean",
        "foxf1_weighted_bilaterality_index_mean",
        "foxf1_size_note",
        "foxf1_posterior_position_note",
        "foxf1_bilaterality_note",
        "foxf1_weighted_bilaterality_note",
        "mesp2_area_um2_mean",
        "mesp2_fraction_mean",
        "mesp2_bilaterality_index_mean",
        "mesp2_weighted_bilaterality_index_mean",
        "mesp2_size_note",
        "mesp2_bilaterality_note",
        "mesp2_weighted_bilaterality_note",
        "pax8_area_um2_mean",
        "pax8_fraction_mean",
        "pax8_bilaterality_index_mean",
        "pax8_weighted_bilaterality_index_mean",
        "pax8_size_note",
        "pax8_bilaterality_note",
        "pax8_weighted_bilaterality_note",
        "draft_day5_description",
        "bilateral somites (y/n)",
        "neural tube with lumen (y/n)",
    ]
].sort_values(["trunk_morph_id"]).reset_index(drop=True)

HANDOFF_TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
handoff_df.to_csv(HANDOFF_TABLE_PATH, sep="\t", index=False)
handoff_df.to_excel(HANDOFF_XLSX_PATH, index=False)

print(f"Wrote {HANDOFF_TABLE_PATH.relative_to(ROOT).as_posix()}")
print(f"Wrote {HANDOFF_XLSX_PATH.relative_to(ROOT).as_posix()}")
display(Markdown("### Inline day-5 phenotype handoff table"))
display(
    handoff_df.style.hide(axis="index").format(
        {
            "length_width_ratio_mean": "{:.2f}",
            "major_axis_length_um_mean": "{:.1f}",
            "minor_axis_length_um_mean": "{:.1f}",
            "consensus_axis_length_um": "{:.1f}",
            "consensus_axis_perp_width_um_mean": "{:.1f}",
            "consensus_axis_length_width_ratio_mean": "{:.2f}",
            "foxf1_area_um2_mean": "{:.1f}",
            "foxf1_fraction_mean": "{:.3f}",
            "foxf1_posterior_edge_distance_um_mean": "{:.1f}",
            "foxf1_posterior_p10_distance_um_mean": "{:.1f}",
            "foxf1_posterior_centroid_distance_um_mean": "{:.1f}",
            "foxf1_bilaterality_index_mean": "{:.2f}",
            "foxf1_weighted_bilaterality_index_mean": "{:.2f}",
            "mesp2_area_um2_mean": "{:.1f}",
            "mesp2_fraction_mean": "{:.3f}",
            "mesp2_bilaterality_index_mean": "{:.2f}",
            "mesp2_weighted_bilaterality_index_mean": "{:.2f}",
            "pax8_area_um2_mean": "{:.1f}",
            "pax8_fraction_mean": "{:.3f}",
            "pax8_bilaterality_index_mean": "{:.2f}",
            "pax8_weighted_bilaterality_index_mean": "{:.2f}",
        }
    )
)


## Next Step

If this rough draft table looks useful, the next pass can pair each row with representative day-5 images so the descriptive columns can be visually verified before handoff.
